# Player Lifecycle & Cross-Sector Analytics

Комплексний ноутбук для аналізу поведінки, прогнозування та канальної ефективності гравців вересня.


## 1. План дослідження
1. Імпорт бібліотек та перевірка залежностей
2. Завантаження/імітація даних і первинна діагностика
3. Препроцесинг та побудова повної сітки (user × date × sector)
4. Ознаки: RFM, лаги, rolling, behavioral, statistical, contextual
5. Поведінковий аналіз та матриці переходів
6. Кластеризація користувачів
7. Прогностичні моделі (класифікація/регресія/активація)
8. Sequence mining та аномалії
9. Канальна ефективність
10. Бізнес-інсайти


## 2. Імпорт бібліотек


In [ ]:
import importlib
import warnings
warnings.filterwarnings('ignore')

REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'sklearn': 'sklearn',
    'lightgbm': 'lightgbm',
    'mlxtend': 'mlxtend',
    'hmmlearn': 'hmmlearn',
    'ruptures': 'ruptures',
    'shap': 'shap'
}
missing = [pkg for pkg, module in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print('⚠️ Відсутні пакети:', ', '.join(missing))
    print('Встановіть за потреби через %pip install ' + ' '.join(missing))
else:
    print('✅ Усі пакети наявні')


**Детально про перевірку залежностей**
**Навіщо:** перед запуском аналізу пересвідчитися, що всі бібліотеки присутні.
**Як це працює:**
1. Імпортуємо `importlib` для перевірки модулів та `warnings` для приглушення попереджень.
2. У словнику `REQUIRED` зазначено ключові пакети (numpy, pandas, seaborn, matplotlib, scipy, sklearn, lightgbm, mlxtend, hmmlearn, ruptures, shap).
3. Вираз `importlib.util.find_spec(module)` повертає `None`, якщо пакет не знайдено; comprehension будує список `missing`.
4. Якщо список порожній — друкуємо «✅ Усі пакети наявні»; інакше показуємо перелік відсутніх та команду `%pip install`.
**Очікуваний результат:** текстове повідомлення виводиться одразу після запуску комірки.
**Міні-приклад:** якщо у вашому середовищі немає `shap`, ви побачите «⚠️ Відсутні пакети: shap» і підказку для встановлення.


In [ ]:
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report, roc_auc_score, average_precision_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import IsolationForest

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth
except Exception:
    TransactionEncoder = None
    fpgrowth = None

try:
    from hmmlearn import hmm
except Exception:
    hmm = None

try:
    import ruptures as rpt
except Exception:
    rpt = None

try:
    import shap
except Exception:
    shap = None

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(7)


**Детально про імпорт бібліотек**
**Призначення:** зібрати всі інструменти, які будуть потрібні далі.
**Пояснення рядків:**
1. `Path`, `List`, `Dict` — робота з файлами та підказки типів.
2. `numpy`, `pandas`, `seaborn`, `matplotlib` — основні інструменти для обробки та візуалізації.
3. Імпорт із `scipy` та `sklearn` покриває ентропію, стандартизацію, PCA, KMeans, метрики, TimeSeriesSplit, IsolationForest.
4. Блоки `try/except` для `lightgbm`, `mlxtend`, `hmmlearn`, `ruptures`, `shap` роблять ноутбук стійким: якщо пакета нема, відповідна змінна отримує `None`, і нижче код пропустить такі кроки.
5. Налаштування стилю `sns.set_theme` та `plt.rcParams` забезпечують однаковий вигляд графіків; `np.random.seed(7)` гарантує відтворюваність синтетики.
**Приклад:** якщо `lightgbm` не встановлено, пізніше виведеться повідомлення «LightGBM недоступний», але ноутбук все одно виконуватиметься.


## 3. Завантаження даних
- Якщо існує `data/transactions.csv`, читаємо його.
- Інакше генеруємо синтетичний датасет (30 днів, 200 користувачів).


In [ ]:
DATA_PATH = Path('data/transactions.csv')
SECTORS = ['virtual', 'other', 'tv']
CHANNELS = ['organic', 'paid_search', 'paid_social', 'affiliate']

if DATA_PATH.exists():
    raw_df = pd.read_csv(DATA_PATH)
    origin = 'source'
else:
    start_date = pd.Timestamp('2024-09-01')
    dates = pd.date_range(start_date, periods=30, freq='D')
    users = np.arange(10_000, 10_200)
    rows = []
    for user in users:
        channel = np.random.choice(CHANNELS, p=[0.35, 0.25, 0.25, 0.15])
        active_days = np.random.choice(dates, size=np.random.randint(6, 18), replace=False)
        for day in active_days:
            for sector in SECTORS:
                engage = (sector == 'virtual') or (np.random.rand() < 0.45)
                if not engage:
                    continue
                spend = np.random.gamma(shape=2.0, scale=5.0)
                if np.random.rand() < 0.25:
                    spend = 0.0
                rows.append({
                    'user_id': user,
                    'date': day.strftime('%Y-%m-%d'),
                    'spend': round(float(spend), 2),
                    'sector': sector,
                    'acquisition_channel': channel
                })
    raw_df = pd.DataFrame(rows)
    origin = 'synthetic'

print('Dataset origin:', origin)
print('Rows:', len(raw_df))
raw_df.head()


**Детально про завантаження/створення даних**
**Що робить:** шукає файл `data/transactions.csv`; якщо його нема, генерує синтетичний датасет.
**Кроки:**
1. Оголошуються базові константи: шлях до файлу, список секторів, список каналів.
2. `if DATA_PATH.exists()` — якщо файл є, читаємо через `pd.read_csv` і помічаємо `origin='source'`.
3. Якщо файлу нема, створюється дата-діапазон на 30 днів (вересень 2024) та список 200 користувачів.
4. Цикл по користувачах вибирає канал (визначені ймовірності) і випадкові активні дні.
5. Внутрішній цикл по секторах вирішує, чи взаємодіє користувач із сектором у той день (`engage`) і генерує витрати з розподілу ґамма; 25% шанс поставити 0, щоб моделювати «зайшов, але нічого не витратив».
6. Усі рядки збираються у список `rows`, потім у DataFrame `raw_df`.
7. Завершення: друк `Dataset origin`, кількості рядків та прев’ю `head()`.
**Приклад:** якщо файл відсутній, ви побачите `Dataset origin: synthetic` і перші 5 рядків із колонками `user_id`, `date`, `spend`, `sector`, `acquisition_channel`.


In [ ]:
raw_df.info()
raw_df.describe(include='all')


**Детально про діагностику даних**
**Мета:** швидко оцінити, чи правильно зчитались дані.
**Що відбувається:**
1. `raw_df.info()` показує кількість рядків, типи колонок і кількість пропусків — одразу видно, чи `date` у правильному форматі, чи є NaN у `spend`.
2. `raw_df.describe(include='all')` дає статистику: мін/макс дат, середні витрати, список унікальних секторів/каналів.
**Очікуване виведення:** дві текстові таблиці у консоль ноутбука.
**Приклад:** для синтетичного набору `spend` має середнє ~7-8, а `unique` секторів = 3 (virtual/other/tv). Якщо бачимо `object` для `date`, значить треба повертатися до препроцесингу.


## 4. Препроцесинг та побудова сітки


In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['sector'] = df['sector'].str.lower().str.strip().replace({'sports': 'other', 'tv-show': 'tv'})
    df.loc[~df['sector'].isin(SECTORS), 'sector'] = 'other'
    df['acquisition_channel'] = df['acquisition_channel'].fillna('unknown').str.lower()
    df['spend'] = df['spend'].fillna(0.0).astype(float)
    return (
        df.groupby(['user_id', 'date', 'sector', 'acquisition_channel'], as_index=False)
          .agg(spend=('spend', 'sum'))
    )


def primary_channel(df: pd.DataFrame) -> pd.DataFrame:
    order = {'organic': 0, 'affiliate': 1, 'paid_search': 2, 'paid_social': 3}
    df = df.assign(rank=df['acquisition_channel'].map(order).fillna(9))
    return df.sort_values(['user_id', 'rank', 'date']).groupby('user_id', as_index=False).first()[['user_id', 'acquisition_channel']]


def build_grid(transactions: pd.DataFrame) -> pd.DataFrame:
    min_date, max_date = transactions['date'].min(), transactions['date'].max()
    date_range = pd.date_range(min_date, max_date, freq='D')
    users = transactions['user_id'].unique()
    channels = primary_channel(transactions)
    multi_index = pd.MultiIndex.from_product([users, date_range, SECTORS], names=['user_id', 'date', 'sector'])
    base = multi_index.to_frame(index=False).merge(channels, on='user_id', how='left')
    merged = base.merge(transactions, on=['user_id', 'date', 'sector', 'acquisition_channel'], how='left')
    merged['spend'] = merged['spend'].fillna(0.0)
    return merged.sort_values(['user_id', 'date', 'sector']).reset_index(drop=True)


def daily_summary(grid: pd.DataFrame) -> pd.DataFrame:
    daily = grid.groupby(['user_id', 'date'], as_index=False).agg(total_spend=('spend', 'sum'))
    daily['is_active'] = (daily['total_spend'] > 0).astype(int)
    gap_records = []
    for user, user_df in daily.groupby('user_id'):
        user_df = user_df.sort_values('date').reset_index(drop=True)
        next_active = None
        gaps = []
        for idx in range(len(user_df) - 1, -1, -1):
            current = user_df.loc[idx, 'date']
            if next_active is None:
                gaps.append(np.nan)
            else:
                gaps.append((next_active - current).days)
            if user_df.loc[idx, 'is_active']:
                next_active = current
        user_df['gap_days'] = list(reversed(gaps))
        gap_records.append(user_df)
    result = pd.concat(gap_records, ignore_index=True)
    result['churn_flag'] = (result['gap_days'] > 7).astype(int)
    return result

transactions = preprocess(raw_df)
full_grid = build_grid(transactions)
daily_metrics = daily_summary(full_grid)

print('Transactions:', len(transactions), 'Full grid:', len(full_grid))
full_grid.head()


**Детально про препроцесинг**
**Призначення:** привести сирі транзакції до єдиного формату та побудувати повну сітку днів.
**Розбір:**
1. `preprocess` нормалізує колонки: дати у datetime, назви секторів у нижній регістр (і маппінг `sports→other`, `tv-show→tv`), канали — у нижній регістр і `unknown`, spend — число.
2. Після цього групуємо дублікати по `user_id + date + sector + acquisition_channel` і підсумовуємо витрати.
3. `primary_channel` обирає «перший» канал користувача за пріоритетом (organic > affiliate > paid_search > paid_social) — це потрібно, коли у користувача різні канали.
4. `build_grid` створює повний `MultiIndex` (усі користувачі × усі дати × усі сектори), додає канали та заповнює відсутні витрати нулями, забезпечуючи повну часову сітку.
5. `daily_summary` агрегує витрати за день, позначає активність (`is_active`), рахує `gap_days` до наступної активності та прапор `churn_flag` (>7 днів паузи).
6. Фінальні рядки запускають усі функції, друкують розміри таблиць і показують `head()` для перевірки.
**Приклад:** якщо у сирих даних є транзакція 1 вересня у virtual і 3 вересня у tv, після `build_grid` між ними з’являться рядки з `spend=0`, а `gap_days` покаже, скільки днів користувач чекав до наступної покупки.


## 5. Ознаки


In [ ]:
def compute_features(grid: pd.DataFrame, daily: pd.DataFrame) -> pd.DataFrame:
    df = grid.copy()
    df = df.sort_values(['user_id', 'sector', 'date'])
    df['spend_lag_1'] = df.groupby(['user_id', 'sector'])['spend'].shift(1).fillna(0.0)
    df['spend_lag_7'] = df.groupby(['user_id', 'sector'])['spend'].shift(7).fillna(0.0)
    df['roll_sum_3'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(3, min_periods=1).sum())
    df['roll_mean_7'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(7, min_periods=1).mean())
    df['roll_std_7'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(7, min_periods=1).std().fillna(0.0))

    pivot = df.pivot_table(index=['user_id', 'date'], columns='sector', values='spend', fill_value=0.0)
    for s in SECTORS:
        if s not in pivot.columns:
            pivot[s] = 0.0
    pivot['virtual_other_ratio'] = pivot['virtual'] / (pivot['other'] + 1e-6)
    pivot['virtual_tv_ratio'] = pivot['virtual'] / (pivot['tv'] + 1e-6)
    pivot['other_tv_ratio'] = pivot['other'] / (pivot['tv'] + 1e-6)
    ratio_df = pivot.reset_index()

    rfm_base = df[df['spend'] > 0]
    max_date = df['date'].max()
    rfm = (
        rfm_base.groupby(['user_id', 'sector'])
                .agg(
                    recency=('date', lambda x: (max_date - x.max()).days),
                    frequency=('date', 'nunique'),
                    monetary=('spend', 'sum')
                )
                .reset_index()
    )
    rfm_pivot = rfm.pivot(index='user_id', columns='sector', values=['recency', 'frequency', 'monetary']).fillna(0)
    rfm_pivot.columns = [f"{m}_{s}" for m, s in rfm_pivot.columns]
    rfm_pivot = rfm_pivot.reset_index()

    behavior = []
    for user, user_daily in daily.groupby('user_id'):
        user_daily = user_daily.sort_values('date')
        streak = 0
        streaks = []
        cooldowns = []
        last_active = None
        for _, row in user_daily.iterrows():
            if row['is_active']:
                streak += 1
                if last_active is not None:
                    cooldowns.append((row['date'] - last_active).days)
                last_active = row['date']
            else:
                streak = 0
            streaks.append(streak)
        behavior.append({
            'user_id': user,
            'days_active': int(user_daily['is_active'].sum()),
            'longest_streak': int(max(streaks) if streaks else 0),
            'median_cooldown': float(np.median(cooldowns)) if cooldowns else np.nan,
            'zero_share': 1 - user_daily['is_active'].mean(),
            'avg_gap': user_daily['gap_days'].mean(skipna=True),
            'max_gap': user_daily['gap_days'].max(skipna=True)
        })
    behavior_df = pd.DataFrame(behavior)

    stats = []
    for user, user_df in df.groupby('user_id'):
        spend_by_sector = user_df.groupby('sector')['spend'].sum().reindex(SECTORS, fill_value=0.0)
        total = spend_by_sector.sum()
        probs = (spend_by_sector / total).replace([np.inf, np.nan], 0)
        stats.append({
            'user_id': user,
            'gini_spend': float(1 - np.sum(np.square(probs))) if total > 0 else 0.0,
            'entropy_spend': float(scipy_entropy(probs + 1e-9, base=2)) if total > 0 else 0.0
        })
    stats_df = pd.DataFrame(stats)

    df = df.merge(ratio_df, on=['user_id', 'date'], how='left')
    df = df.merge(daily[['user_id', 'date', 'gap_days', 'churn_flag']], on=['user_id', 'date'], how='left')
    df['gap_days'] = df['gap_days'].fillna(0)
    df['churn_flag'] = df['churn_flag'].fillna(0).astype(int)

    day_ctx = df[['user_id', 'date']].copy()
    day_ctx['day_of_week'] = df['date'].dt.day_name()
    day_ctx['weekday_idx'] = df['date'].dt.weekday
    day_ctx['is_weekend'] = day_ctx['weekday_idx'].isin([5, 6]).astype(int)

    enriched = df.merge(day_ctx, on=['user_id', 'date'], how='left')
    user_features = behavior_df.merge(stats_df, on='user_id', how='left').merge(rfm_pivot, on='user_id', how='left')
    return enriched, user_features

feature_daily, user_features = compute_features(full_grid, daily_metrics)
feature_daily.head()


**Детально про конвеєр ознак**
**Мета:** створити всі фічі з технічного завдання — лаги, rolling, RFM, поведінкові й статистичні.
**Кроки:**
1. Починаємо з копії `grid`, сортуємо за `user_id`, `sector`, `date`, щоб лаги рахувалися правильно.
2. `groupby(...).shift(1/7)` створює лагові витрати `spend_lag_1`, `spend_lag_7` (0, якщо історії немає).
3. `rolling(3/7)` додає суму, середнє і стандартне відхилення за вікнами, що ловить локальні тренди.
4. `pivot_table` формує широку таблицю з витратами на день, на її основі рахуються крос-секторальні коефіцієнти (`virtual_other_ratio` тощо) з додаванням `1e-6`, щоб уникнути ділення на нуль.
5. RFM: беремо тільки рядки зі spend>0, знаходимо `recency` (дні від останньої активності), `frequency` (активні дні) та `monetary` (сума) по кожному сектору; `pivot` робить їх окремими колонками.
6. Поведінкові метрики: цикл по користувачу обчислює `days_active`, `longest_streak`, `median_cooldown`, `zero_share`, `avg_gap`, `max_gap` на основі `daily`.
7. Статистичні індекси: для кожного користувача рахуємо частки витрат по секторах, з них Gini та ентропію — показник диверсифікації.
8. Об’єднання: зливаємо все в `df`, додаємо `gap_days`, `churn_flag`, календарні ознаки (`day_of_week`, `weekday_idx`, `is_weekend`).
9. Функція повертає `enriched` (покадрові фічі на кожен день і сектор) та `user_features` (агрегації для кластеризації), одразу показується `feature_daily.head()`.
**Приклад:** якщо користувач 10011 витратив 30 у virtual за останні 7 днів і 5 у tv, коефіцієнт `virtual_tv_ratio` ≈ 6; якщо він був неактивний 8 днів, `churn_flag=1` і `median_cooldown` збільшується.


## 6. Матриці переходів та динаміка витрат


In [ ]:
def transition_matrix(grid: pd.DataFrame, weighted: bool = False) -> pd.DataFrame:
    records = []
    for user, user_df in grid.groupby('user_id'):
        daily = user_df.pivot_table(index='date', columns='sector', values='spend', fill_value=0.0)
        daily['state'] = np.where(daily.sum(axis=1) == 0, 'pause', daily[SECTORS].idxmax(axis=1))
        daily['weight'] = daily.sum(axis=1)
        states = daily[['state', 'weight']].reset_index()
        for i in range(len(states) - 1):
            w = states.loc[i + 1, 'weight'] if weighted else 1
            records.append((states.loc[i, 'state'], states.loc[i + 1, 'state'], w))
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records, columns=['from', 'to', 'value'])
    return df.pivot_table(index='from', columns='to', values='value', aggfunc='sum', fill_value=0.0)

trans_counts = transition_matrix(full_grid, weighted=False)
trans_weighted = transition_matrix(full_grid, weighted=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if not trans_counts.empty:
    sns.heatmap(trans_counts, annot=True, fmt='.0f', cmap='Blues', ax=axes[0])
    axes[0].set_title('Transition counts')
else:
    axes[0].set_visible(False)
if not trans_weighted.empty:
    sns.heatmap(trans_weighted, annot=True, fmt='.1f', cmap='Purples', ax=axes[1])
    axes[1].set_title('Weighted transitions')
else:
    axes[1].set_visible(False)
plt.show()

rolling = (full_grid.groupby(['date', 'sector'])['spend'].sum().reset_index())
rolling['roll_mean_7'] = rolling.groupby('sector')['spend'].transform(lambda s: s.rolling(7, min_periods=1).mean())
plt.figure(figsize=(12, 6))
for sector in SECTORS:
    subset = rolling[rolling['sector'] == sector]
    plt.plot(subset['date'], subset['roll_mean_7'], label=f'{sector} 7d mean')
plt.legend(); plt.title('7-day rolling spend by sector'); plt.show()

cooldown_stats = daily_metrics.groupby('user_id').agg(
    avg_gap=('gap_days', 'mean'),
    cooldown_75=('gap_days', lambda x: np.nanpercentile(x, 75)),
    churn_rate=('churn_flag', 'mean'),
    active_days=('is_active', 'sum')
)
cooldown_stats.describe()


**Детально про поведінковий аналіз**
**Призначення:** побачити, як користувачі перемикаються між секторами і як змінюються витрати в часі.
**Розбір коду:**
1. `transition_matrix` проходить по кожному користувачу, визначає щоденний стан (максимальний сектор або `pause`) і накопичує переходи `state → next_state`. Параметр `weighted` дозволяє підсумовувати витрати замість кількості.
2. Після збору переходів вони агрегуються у матрицю (рядки — звідки, стовпці — куди). Якщо даних немає, повертається порожній DataFrame.
3. Далі рахуємо дві версії матриць (`trans_counts`, `trans_weighted`) і малюємо дві теплові карти.
4. `rolling = ...` створює 7-денне ковзне середнє витрат по кожному сектору, яке зручно дивитися як тренд активності.
5. `cooldown_stats` підраховує середні та 75-й перцентиль gap-ів, частку churn (`churn_flag`) та кількість активних днів — це допомагає знайти «вікна реактивації».
**Очікуваний результат:** теплові карти, лінійний графік, таблиця описової статистики за cooldown-ами.
**Приклад:** якщо у матриці видно 120 переходів `virtual → pause` і лише 30 `virtual → tv`, значить треба стимулювати cross-sell; а середній `avg_gap = 3.5` підказує, що промо слід надсилати на третій день без активності.


## 7. Кластеризація


In [ ]:
cluster_input = user_features.fillna(0).set_index('user_id')
scaler = StandardScaler()
scaled = scaler.fit_transform(cluster_input)

sil_scores = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    sil_scores[k] = silhouette_score(scaled, model.fit_predict(scaled))

best_k = max(sil_scores, key=sil_scores.get)
km = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
cluster_labels = km.fit_predict(scaled)
cluster_input['cluster'] = cluster_labels

pca = PCA(n_components=2).fit_transform(scaled)
cluster_input['pc1'] = pca[:, 0]
cluster_input['pc2'] = pca[:, 1]

plt.figure(figsize=(9, 7))
sns.scatterplot(data=cluster_input, x='pc1', y='pc2', hue='cluster', palette='viridis')
plt.title('PCA projection of clusters')
plt.show()

cluster_summary = cluster_input.groupby('cluster').agg(
    users=('pc1', 'count'),
    avg_days_active=('days_active', 'mean'),
    avg_zero_share=('zero_share', 'mean'),
    avg_virtual_spend=('monetary_virtual', 'mean'),
    avg_tv_spend=('monetary_tv', 'mean'),
    avg_entropy=('entropy_spend', 'mean')
)
cluster_summary


**Детально про кластеризацію**
**Ціль:** поділити користувачів на типи поведінки.
**Етапи:**
1. `cluster_input = user_features.fillna(0)` наповнює пропуски нулями (щоб алгоритм не падав) та ставить `user_id` у індекс.
2. `StandardScaler` масштабує всі ознаки — важливо, бо `days_active` та `entropy` мають різні одиниці.
3. Цикл `for k in range(2,7)` тренує KMeans для 2-6 кластерів і рахує `silhouette_score`, щоб підібрати оптимальне `k`.
4. `best_k = max(...)` вибирає кластеризацію з найвищим силуетом; далі запускається фінальний KMeans.
5. PCA з 2 компонентами робить проєкцію у площину для наочності, і ми малюємо розсіяння за допомогою seaborn.
6. `cluster_summary` агрегує ключові метрики по кластерах: скільки користувачів, середні дні активності, середні витрати по virtual/tv, рівномірність витрат (`entropy`).
**Вихід:** інтерактивний графік та таблиця `cluster_summary` під ним.
**Приклад:** часто силует підказує `k=4`, тоді можна лейблити кластери як Steady, Switchers, Sprinters, Entertainment-first на основі метрик з `cluster_summary`.


## 8. Прогностичні моделі


In [ ]:
def modeling_frame(df: pd.DataFrame) -> pd.DataFrame:
    pivot = df.pivot_table(index=['user_id', 'date'], columns='sector', values='spend', fill_value=0.0).reset_index()
    features = df.drop_duplicates(['user_id', 'date'])[['user_id', 'date', 'virtual_other_ratio', 'virtual_tv_ratio', 'other_tv_ratio', 'gap_days', 'churn_flag']]
    merged = pivot.merge(features, on=['user_id', 'date'], how='left')
    merged['next_date'] = merged.groupby('user_id')['date'].shift(-1)
    for sector in SECTORS:
        merged[f'next_{sector}'] = merged.groupby('user_id')[sector].shift(-1)
    merged['next_sector'] = merged[[f'next_{s}' for s in SECTORS]].idxmax(axis=1).str.replace('next_', '')
    merged['next_spend'] = merged[[f'next_{s}' for s in SECTORS]].sum(axis=1)
    merged = merged.dropna(subset=['next_date'])
    merged = pd.get_dummies(merged, columns=['churn_flag'], drop_first=False)
    merged = pd.get_dummies(merged, columns=[], drop_first=False)
    return merged

model_df = modeling_frame(feature_daily)
model_df.head()


**Детально про формування датафрейму для моделей**
**Мета:** підготувати навчальну вибірку, де кожний рядок відповідає парі користувач-день та містить значення наступного дня.
**Алгоритм:**
1. `pivot_table` перетворює дані у форму широку: у кожного рядка — витрати по `virtual`, `other`, `tv` за поточну дату.
2. `features = df.drop_duplicates([...])` залишає поведінкові коефіцієнти та ознаки gap/churn на день.
3. `merged = pivot.merge(features, ...)` поєднує ці дві частини.
4. `.groupby('user_id')['date'].shift(-1)` записує дату наступного дня у `next_date`.
5. Для кожного сектору створюється колонка `next_sector` з витратами наступного дня; `.idxmax()` визначає, який сектор був головним.
6. `next_spend` — сумарні витрати наступного дня, що стане ціллю для регресії.
7. `dropna(subset=['next_date'])` прибирає останні дні (бо для них немає «завтра»).
8. `pd.get_dummies` залишено для можливого one-hot кодування (фактично зберігає `churn_flag_0/1`).
**Результат:** DataFrame `model_df` з >10 колонками (включно з лагами, rolling, ratio, gap, churn, day-of-week), який одразу показується `head()`.
**Приклад:** для користувача 10023 з витратами 12 у virtual сьогодні та 5 у tv завтра отримаємо `next_sector='tv'`, `next_spend=5` та лагові фічі з попередніх днів.


In [ ]:
def train_classifier(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend'])
    y = df['next_sector']
    mask = y != 'nan'
    X, y = X[mask], y[mask]
    splits = TimeSeriesSplit(n_splits=3)
    reports = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        lgb_train = lgb.Dataset(train_x, label=train_y)
        lgb_val = lgb.Dataset(test_x, label=test_y, reference=lgb_train)
        params = {
            'objective': 'multiclass',
            'num_class': len(y.unique()),
            'learning_rate': 0.05,
            'metric': 'multi_logloss',
            'num_leaves': 31,
            'verbose': -1
        }
        model = lgb.train(params, lgb_train, valid_sets=[lgb_val], num_boost_round=400, early_stopping_rounds=40, verbose_eval=False)
        preds = np.argmax(model.predict(test_x), axis=1)
        print(f'Fold {fold}
', classification_report(test_y, preds))
        reports.append(model)
    return reports[-1] if reports else None


def train_regressor(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend'])
    y = df['next_spend']
    splits = TimeSeriesSplit(n_splits=3)
    metrics = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        lgb_train = lgb.Dataset(train_x, label=train_y)
        lgb_val = lgb.Dataset(test_x, label=test_y, reference=lgb_train)
        params = {
            'objective': 'regression',
            'metric': ['l1', 'l2'],
            'learning_rate': 0.05,
            'num_leaves': 31,
            'feature_fraction': 0.9,
            'verbose': -1
        }
        model = lgb.train(params, lgb_train, valid_sets=[lgb_val], num_boost_round=400, early_stopping_rounds=40, verbose_eval=False)
        preds = model.predict(test_x)
        mae = mean_absolute_error(test_y, preds)
        rmse = mean_squared_error(test_y, preds, squared=False)
        metrics.append({'fold': fold, 'MAE': mae, 'RMSE': rmse})
        print(f'Fold {fold}: MAE={mae:.3f}, RMSE={rmse:.3f}')
    return metrics

clf_model = train_classifier(model_df) if lgb is not None else None
reg_metrics = train_regressor(model_df) if lgb is not None else None
reg_metrics


**Детально про прогнозування сектору та витрат**
**Завдання:** навчити дві моделі LightGBM — класифікатор сектору та регресор суми витрат на наступний день.
**Хід виконання:**
1. Перша функція `train_classifier` відкидає службові колонки та використовує `next_sector` як ціль. Маска `y != 'nan'` прибирає порожні записи (таке може трапитись на краях серії).
2. `TimeSeriesSplit` із 3 фолдами забезпечує walk-forward валідацію.
3. На кожному фолді створюється `lgb.Dataset`, задаються параметри мультикласу (`num_class` = кількість секторів), тренується модель з ранньою зупинкою, а `classification_report` дає precision/recall/F1 по кожному сектору.
4. Функція повертає останню модель, яку можна зберегти або використати для SHAP.
5. `train_regressor` працює аналогічно, але оптимізує MAE/RMSE. Параметри — `objective='regression'`, `metric=['l1','l2']`.
6. Для кожного фолду друкується `MAE` та `RMSE`, а словники додаються до списку `metrics`.
7. Заключні рядки викликають обидві функції (якщо LightGBM встановлено) і друкують метрики регресії.
**Очікуваний результат:** текстові звіти у консолі ноутбука. Наприклад, `Fold 2: MAE=3.214, RMSE=4.987` показує середню похибку в грошовому еквіваленті.
**Приклад:** якщо модель класифікації дає F1=0.6 для `tv`, це означає, що 60% передбачень `tv` точні, що достатньо для таргетованих пропозицій.


In [ ]:
def activation_model(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    df = df.copy()
    df['activate_other_tv'] = (df['next_other'] + df['next_tv'] > 0).astype(int)
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend', 'activate_other_tv'])
    y = df['activate_other_tv']
    splits = TimeSeriesSplit(n_splits=3)
    metrics = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        train_set = lgb.Dataset(train_x, label=train_y)
        val_set = lgb.Dataset(test_x, label=test_y, reference=train_set)
        params = {'objective': 'binary', 'metric': ['auc', 'binary_logloss'], 'learning_rate': 0.05, 'num_leaves': 31}
        model = lgb.train(params, train_set, valid_sets=[val_set], num_boost_round=300, early_stopping_rounds=30, verbose_eval=False)
        preds = model.predict(test_x)
        metrics.append({'fold': fold, 'ROC-AUC': roc_auc_score(test_y, preds), 'PR-AUC': average_precision_score(test_y, preds)})
    return pd.DataFrame(metrics)

activation_stats = activation_model(model_df) if lgb is not None else None
activation_stats


**Детально про модель крос-секторальної активації**
**Призначення:** спрогнозувати, чи користувач наступного дня витратить щось у `other` або `tv` після активності у `virtual`.
**Процес:**
1. Перевірка `if lgb is None` гарантує, що код не впаде без LightGBM (тоді просто друкується попередження).
2. Створюється мітка `activate_other_tv`, що дорівнює 1, якщо сумарні витрати в `other` + `tv` наступного дня > 0.
3. Вхідні фічі — усі лаги, ролінги, поведінкові метрики, коефіцієнти тощо (усе, що залишилось після `drop`).
4. `TimeSeriesSplit` ідеально підходить для часових рядів: щоразу тренування відбувається на ранніх днях, а тест — на пізніших.
5. Всередині циклу збираємо `lgb.Dataset`, задаємо гіперпараметри (`objective='binary'`, `num_leaves=31`), тренуємо з early stopping.
6. Для кожного фолду обчислюємо ROC-AUC та PR-AUC — ці метрики особливо важливі для моделей активації.
**Результат:** DataFrame `activation_stats` з рядком на кожен фолд; чим вищі значення (наближені до 1), тим краще модель відрізняє майбутню активацію.
**Приклад:** якщо у фолді 1 ROC-AUC = 0.78, значить у 78% пар позитив/негатив модель вгадує, хто активується.


## 9. Sequence mining та аномалії


In [ ]:
def build_sequences(grid: pd.DataFrame) -> List[List[str]]:
    sequences = []
    for _, user_df in grid.groupby('user_id'):
        daily = user_df.pivot_table(index='date', columns='sector', values='spend', fill_value=0.0)
        sequence = daily.apply(lambda row: 'pause' if row.sum() == 0 else row[SECTORS].idxmax(), axis=1).tolist()
        sequences.append(sequence)
    return sequences

sequences = build_sequences(full_grid)
print('Total sequences:', len(sequences))

if TransactionEncoder is not None and fpgrowth is not None:
    te = TransactionEncoder()
    tx = te.fit_transform([list(dict.fromkeys(seq)) for seq in sequences])
    freq = fpgrowth(pd.DataFrame(tx, columns=te.columns_), min_support=0.1, use_colnames=True)
    freq.sort_values('support', ascending=False).head()
else:
    print('mlxtend недоступний')

if hmm is not None:
    mapping = {'virtual': 0, 'other': 1, 'tv': 2, 'pause': 3}
    concatenated = []
    lengths = []
    for seq in sequences:
        concatenated.extend([mapping[s] for s in seq])
        lengths.append(len(seq))
    model = hmm.MultinomialHMM(n_components=4, random_state=42, n_iter=100)
    model.fit(np.array(concatenated).reshape(-1, 1), lengths)
    print('HMM transitions:
', model.transmat_)
else:
    print('hmmlearn недоступний')

if rpt is not None:
    total_spend = full_grid.groupby('date')['spend'].sum().reset_index()
    algo = rpt.Pelt(model='rbf').fit(total_spend['spend'].values)
    cp = algo.predict(pen=5)
    print('Change points:', cp)
else:
    print('ruptures недоступний')

iso = IsolationForest(contamination=0.02, random_state=42)
user_daily = full_grid.groupby(['user_id', 'date'])['spend'].sum().reset_index()
user_daily['anomaly_score'] = iso.fit_predict(user_daily[['spend']])
user_daily[user_daily['anomaly_score'] == -1].head()


**Детально про sequence mining та аномалії**
**Що робить:** збирає послідовності станів для кожного користувача і застосовує кілька методів пошуку патернів.
**Кроки:**
1. `build_sequences` перетворює таблицю у списки станів (`virtual`, `other`, `tv`, `pause`) у порядку дат.
2. Якщо доступний `mlxtend`, запускається `fpgrowth`, щоб знайти часті набори секторів за день (напр., {virtual, tv}).
3. Якщо встановлено `hmmlearn`, тренується `MultinomialHMM` на всіх послідовностях — з `model.transmat_` можна прочитати ймовірності переходів (наприклад, `virtual → pause`).
4. Якщо є `ruptures`, виконується пошук changepoint-ів у сумарному spend по днях, показуючи різкі переломи тренду.
5. `IsolationForest` працює як детектор аномалій у денних витратах користувача; рядки з `anomaly_score == -1` — потенційні сплески або шахрайство.
**Очікуваний результат:** друк кількості послідовностей, таблиці частих наборів (якщо є), матриці переходів HMM, список change points та попередній перегляд аномальних днів.
**Міні-приклад:** у синтетичному наборі часто з’являється патерн `virtual → pause → tv`. Якщо `IsolationForest` знаходить день зі spend > 40 у порівнянні з типовими 10, він з’явиться у `head()` аномалій.


## 10. Канальна ефективність


In [ ]:
channel_perf = feature_daily.groupby('acquisition_channel').agg(
    users=('user_id', 'nunique'),
    active_share=('spend', lambda x: (x > 0).mean()),
    avg_spend=('spend', 'mean'),
    avg_gap=('gap_days', 'mean'),
    median_gap=('gap_days', 'median'),
    virtual_focus=('virtual', 'mean') if 'virtual' in feature_daily.columns else ('spend', 'mean'),
    ratio_virtual_other=('virtual_other_ratio', 'mean'),
    ratio_virtual_tv=('virtual_tv_ratio', 'mean')
).reset_index()
channel_perf


**Детально про таблицю ефективності каналів**
**Призначення:** консолідувати ключові KPI по каналах залучення.
**Як працює:**
1. `groupby('acquisition_channel')` збирає всі дні й користувачів по кожному каналу.
2. `users=('user_id', 'nunique')` рахує, скільки унікальних користувачів прийшло з каналу — це база для часток.
3. `active_share` підраховує, у яку частину днів користувачі були активні (>=1 транзакції) — показник утримання.
4. `avg_spend` і `avg_gap`/`median_gap` вимірюють монетизацію та «охолодження» між сесіями.
5. `virtual_focus` показує частку витрат саме у virtual (або fallback на `spend` якщо колонки немає).
6. `ratio_virtual_other` та `ratio_virtual_tv` дають середні крос-секторальні коефіцієнти.
**Вихід:** DataFrame `channel_perf` виводиться прямо під кодом, і його легко прокрутити у ноутбуці.
**Приклад інтерпретації:** якщо `affiliate` має `active_share = 0.65`, `avg_gap = 2`, а `ratio_virtual_tv = 1.8`, то такі користувачі швидше повертаються і частіше переходять у TV після virtual.


In [ ]:
cluster_mix = cluster_input.reset_index().merge(primary_channel(transactions), on='user_id', how='left')
cluster_mix = cluster_mix.groupby(['acquisition_channel', 'cluster']).size().reset_index(name='count')

plt.figure(figsize=(10, 4))
sns.barplot(data=channel_perf, x='acquisition_channel', y='avg_spend')
plt.title('Average spend per channel')
plt.show()

pivot = cluster_mix.pivot(index='acquisition_channel', columns='cluster', values='count').fillna(0)
(pivot.T / pivot.sum(axis=1)).T.plot(kind='bar', stacked=True, figsize=(10, 4))
plt.ylabel('Cluster share')
plt.title('Cluster composition by channel')
plt.show()

if not trans_counts.empty:
    for channel, channel_df in full_grid.groupby('acquisition_channel'):
        matrix = transition_matrix(channel_df, weighted=False)
        if matrix.empty:
            continue
        plt.figure(figsize=(6, 4))
        sns.heatmap(matrix, annot=True, fmt='.0f', cmap='Greens')
        plt.title(f'Transitions — {channel}')
        plt.show()


**Детально про візуалізацію каналів**
**Що робить блок:** будує два графіки (середні витрати та склад кластерів) і додаткові теплові карти переходів у розрізі каналів.
**Кроки:**
1. `cluster_mix` зливає кластерні мітки з базовим каналом, щоб знати, скільки користувачів кожного типу прийшло з конкретного каналу.
2. `sns.barplot` показує середній чек (`avg_spend`) по каналах — це відповідає на питання, які канали дають найбільш цінних клієнтів.
3. Стеки `plot(kind='bar', stacked=True)` відображають частку кожного кластера у каналі, допомагаючи побачити, де більше Steady/Sprinters тощо.
4. Цикл по `full_grid.groupby('acquisition_channel')` викликає `transition_matrix`, малює теплові карти переходів для кожного каналу окремо — так видно, які канали генерують найбільше крос-секторальних міграцій.
**Очікуваний результат:** набір графіків у ноутбуці. Наприклад, якщо `paid_search` має високий стовпчик, це означає вищі середні витрати, а у тепловій карті можна побачити, що з `virtual` вони частіше йдуть у `tv`.
**Міні-приклад:** для синтетичних даних зазвичай `virtual → pause` домінує, а у каналі `organic` буде більше Steady spenders, тоді як `paid_social` може мати більше Switchers.


## 11. Бізнес-інсайти
- **Траєкторії**: матриці переходів показують, наскільки часто virtual → pause → tv та інші маршрути; це визначає вікна реактивації.
- **Типи гравців**: кластери приблизно відповідають `Steady Spenders`, `Cross-market Switchers`, `Sprinters`, `Entertainment-first`.
- **Життєвий цикл**: метрики streak/gap/zero_share підказують, коли користувачі «остивають» та коли їх варто активувати промо.
- **Прогноз**: моделі LightGBM (якщо доступні) оцінюють сектор і spend наступного дня, а також cross-sector activation.
- **Аномалії**: IsolationForest/ruptures/HMM дозволяють виявити різкі зміни та потенційно ризикових користувачів.
- **Канали**: агреговані метрики показують, які acquisition-канали дають вищий LTV, довші активні періоди та багатші переходи між секторами.

Подальші кроки: інтегрувати реальні дані, тюнити моделі, підготувати рекомендації для retention та крос-продажів.
